In [27]:
from pathlib import Path
import time
import numpy as np
from IPython.display import clear_output
import mujoco
import mujoco.viewer
import inspect

In [36]:
MJCF_PATH = Path("/workspace/mechanical/FreeCAD/bala2-fire/bala2-fire-simplified.xml")
MOTOR_SPEED_STEP = 0.1
MOTOR_SPEED_LIMIT = 1.0
PRINT_EVERY = 50

LEFT_MOTOR = "left_motor"
RIGHT_MOTOR = "right_motor"
CHASSIS_BODY = "chassis"

In [37]:
ctrl = {"left": 0.0, "right": 0.0}

In [38]:
KEY_BACKSPACE = 259
KEY_UP = 265
KEY_DOWN = 264
KEY_LEFT  = 263
KEY_RIGHT = 262

def clamp(x):
    return max(-MOTOR_SPEED_LIMIT, min(MOTOR_SPEED_LIMIT, x))

def key_callback(keycode):
    global cf_pitch
    if keycode == KEY_BACKSPACE:
        ctrl["left"] = ctrl["right"] = 0.0
        cf_pitch = 0.0
        mujoco.mj_resetData(model, data)
    elif keycode == KEY_UP:
        ctrl["left"]  = clamp(ctrl["left"]  + MOTOR_SPEED_STEP)
        ctrl["right"] = clamp(ctrl["right"] + MOTOR_SPEED_STEP)
    elif keycode == KEY_DOWN:
        ctrl["left"]  = clamp(ctrl["left"]  - MOTOR_SPEED_STEP)
        ctrl["right"] = clamp(ctrl["right"] - MOTOR_SPEED_STEP)
    elif keycode == KEY_LEFT:
        ctrl["left"]  = clamp(ctrl["left"]  - MOTOR_SPEED_STEP)
        ctrl["right"] = clamp(ctrl["right"] + MOTOR_SPEED_STEP)
    elif keycode == KEY_RIGHT:
        ctrl["left"]  = clamp(ctrl["left"]  + MOTOR_SPEED_STEP)
        ctrl["right"] = clamp(ctrl["right"] - MOTOR_SPEED_STEP)

In [47]:
# Complementary filter state
cf_pitch = 0.0
cf_alpha = 0.99

# PD controller gains (tune these if it doesn't balance well)
KP = 15.0
KD = 0.5

def get_pitch(data, model):
    """Estimate pitch using complementary filter on IMU data."""
    global cf_pitch
    accel = data.sensor("imu_accel").data
    gyro  = data.sensor("imu_gyro").data
    pitch_rate = gyro[1]
    accel_pitch = -np.arctan2(accel[0], accel[2])
    cf_pitch = cf_alpha * (cf_pitch + pitch_rate * model.opt.timestep) + \
               (1 - cf_alpha) * accel_pitch
    return cf_pitch, pitch_rate

def balance_action(pitch, pitch_rate, drive_cmd=0.0):
    """PD controller output to keep robot upright, plus a drive command offset."""
    action = (KP * pitch + KD * pitch_rate) + drive_cmd
    return float(np.clip(action, -1.0, 1.0))

In [48]:
model = mujoco.MjModel.from_xml_path(str(MJCF_PATH))
data  = mujoco.MjData(model)

left_motor_id  = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_ACTUATOR, LEFT_MOTOR)
right_motor_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_ACTUATOR, RIGHT_MOTOR)
chassis_id     = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, CHASSIS_BODY)

print(f"Left motor ID:  {left_motor_id}")
print(f"Right motor ID: {right_motor_id}")
print(f"Chassis body ID: {chassis_id}")

Left motor ID:  0
Right motor ID: 1
Chassis body ID: 1


In [49]:
mujoco.mj_resetData(model, data)
ctrl["left"] = ctrl["right"] = 0.0

steps = 0
with mujoco.viewer.launch_passive(model, data, key_callback=key_callback) as viewer:
    viewer.cam.type      = mujoco.mjtCamera.mjCAMERA_FREE
    viewer.cam.lookat[:] = [0, 0, 0.05]
    viewer.cam.distance  = 0.8
    viewer.cam.azimuth   = 45
    viewer.cam.elevation = -25

    while viewer.is_running():
        step_start = time.time()

        # Get pitch estimate
        pitch, pitch_rate = get_pitch(data, model)
        
        # Balance + drive command (drive_cmd offsets the balance point to move forward/back)
        drive_cmd = -(ctrl["left"] + ctrl["right"]) / 2.0  # symmetric = forward/back
        turn_cmd  = (ctrl["right"] - ctrl["left"]) / 2.0  # asymmetric = turn
        
        motor_base = balance_action(pitch, pitch_rate, drive_cmd)
        data.ctrl[left_motor_id]  = np.clip(motor_base - turn_cmd, -1.0, 1.0)
        data.ctrl[right_motor_id] = np.clip(motor_base + turn_cmd, -1.0, 1.0)

        mujoco.mj_step(model, data)
        viewer.sync()

        steps += 1
        if steps % PRINT_EVERY == 0:
            # Chassis rotation matrix and velocity
            xmat     = data.xmat[chassis_id].reshape(3, 3)
            cvel     = data.cvel[chassis_id]
            world_vel = cvel[3:6]  # linear component of com-based velocity

            # Project world-frame velocity onto each chassis local axis
            vel_x = float(np.dot(world_vel, xmat[:, 0]))  # local X
            vel_y = float(np.dot(world_vel, xmat[:, 1]))  # local Y
            vel_z = float(np.dot(world_vel, xmat[:, 2]))  # local Z

            # Wheel velocities
            qvel_left  = data.qvel[6]
            qvel_right = data.qvel[7]
            wheel_avg  = (qvel_left + qvel_right) / 2.0

            clear_output(wait=True)
            print(f"Motor commands:  left={ctrl['left']:.1f}  right={ctrl['right']:.1f}")
            print()
            print(f"qvel[6] (left):  {qvel_left:.4f} rad/s")
            print(f"qvel[7] (right): {qvel_right:.4f} rad/s")
            print(f"wheel avg:       {wheel_avg:.4f} rad/s")
            print()
            print(f"world_vel:       {world_vel}")
            print()
            print(f"xmat col 0 (X):  {xmat[:, 0]}")
            print(f"xmat col 1 (Y):  {xmat[:, 1]}")
            print(f"xmat col 2 (Z):  {xmat[:, 2]}")
            print()
            print(f"projected vel X: {vel_x:.4f} m/s")
            print(f"projected vel Y: {vel_y:.4f} m/s")
            print(f"projected vel Z: {vel_z:.4f} m/s")
            print()
            print(f"pitch: {pitch:.4f} rad  motor_base: {motor_base:.4f}")
            print(f"pitch_rate: {pitch_rate:.4f} rad/s")
            print(f"motor_base: {motor_base:.4f}")
            print(f"drive_cmd: {drive_cmd:.4f}")
            print(f"left_ctrl: {data.ctrl[left_motor_id]:.4f}")
            print(f"right_ctrl: {data.ctrl[right_motor_id]:.4f}")

        slack = model.opt.timestep - (time.time() - step_start)
        if slack > 0:
            time.sleep(slack)

Motor commands:  left=0.0  right=0.0

qvel[6] (left):  49.0000 rad/s
qvel[7] (right): 49.0000 rad/s
wheel avg:       49.0000 rad/s

world_vel:       [-7.31508429e-16  9.41026652e-18 -1.38970332e-16]

xmat col 0 (X):  [ 7.14736492e-04 -9.74317477e-04  9.99999270e-01]
xmat col 1 (Y):  [-4.59770470e-03  9.99988953e-01  9.77593574e-04]
xmat col 2 (Z):  [-9.99989175e-01 -4.59840006e-03  7.10248972e-04]

projected vel X: -0.0000 m/s
projected vel Y: 0.0000 m/s
projected vel Z: 0.0000 m/s

pitch: -1.5701 rad  motor_base: 1.0000
pitch_rate: -0.0000 rad/s
motor_base: 1.0000
drive_cmd: -0.0000
left_ctrl: 1.0000
right_ctrl: 1.0000


KeyboardInterrupt: 

In [25]:
print(inspect.getsource(balance_action))

def balance_action(pitch, pitch_rate, drive_cmd=0.0):
    """PD controller output to keep robot upright, plus a drive command offset."""
    action = (KP * pitch + KD * pitch_rate) + drive_cmd
    return float(np.clip(action, -1.0, 1.0))

